In [ ]:
import numpy as np
import pandas as pd
import MDAnalysis as mda
from scipy.spatial import Voronoi, ConvexHull, KDTree, Delaunay
from scipy.spatial.distance import cdist
import networkx as nx
import copy
import os
import subprocess
import tempfile
from collections import defaultdict
import multiprocessing as mp
from functools import partial
import concurrent.futures

class ProteinTunnelAnalyzer:
    """
    Analyzes protein tunnels using Voronoi diagrams similar to Caver 3.0.
    Identifies tunnels from protein interior to bulk solvent.
    """
    
    # Standard van der Waals radii for common atoms (in Angstroms)
    VDW_RADII = {
        'H': 1.20, 'C': 1.70, 'N': 1.55, 'O': 1.52, 'S': 1.80, 'P': 1.80,
        'F': 1.47, 'CL': 1.75, 'BR': 1.85, 'I': 1.98, 'NA': 2.27, 'MG': 1.73,
        'K': 2.75, 'CA': 2.31, 'ZN': 1.39, 'FE': 1.80, 'DEFAULT': 1.70
    }
    
    def __init__(self, universe, selection='protein', probe_radius=3.0, min_bottleneck = 0.5):
        """
        Initialize the tunnel analyzer.
        
        Parameters:
        -----------
        universe : MDAnalysis.Universe
            MDAnalysis universe containing the protein structure
        selection : str
            MDAnalysis selection string for atoms to consider
        probe_radius : float
            Radius of the probe sphere used to define bulk solvent accessibility (in Angstroms)
        """
        self.universe = universe
        self.selection = selection
        self.probe_radius = probe_radius
        self.atoms = universe.select_atoms(selection)
        self.positions = self.atoms.positions
        self.atom_radii = self._get_atom_radii()
        self.min_bottleneck = min_bottleneck
        self.protein_com = self.atoms.center_of_mass()
        self.protein_rgyr = self.atoms.radius_of_gyration()
        
        # Create atom KDTree for faster distance calculations
        self.atom_tree = KDTree(self.positions)
        
        # Results storage
        self.voronoi = None
        self.enhanced_voronoi = None
        self.graph = None
        self.tunnels = []
        self.tunnel_dataframe = None
        self.start_point = None
        self.start_vertex = None
        
    def _get_atom_radii(self):
        """Extract van der Waals radii for each atom in the selection."""
        # Use list comprehension instead of explicit loop for speed
        radii = [self.VDW_RADII.get(atom.element.upper(), self.VDW_RADII['DEFAULT']) 
                 for atom in self.atoms]
        return np.array(radii)
    
    def set_start_point_from_coordinates(self, coordinates):
        """
        Set the tunnel starting point using specified coordinates.
        
        Parameters:
        -----------
        coordinates : array-like
            3D coordinates [x, y, z] of the starting point
        """
        self.start_point = np.array(coordinates)
        return self.start_point
    
    def set_start_point_from_selection(self, selection_string):
        """
        Set the tunnel starting point as the center of a MDAnalysis selection.
        
        Parameters:
        -----------
        selection_string : str
            MDAnalysis selection string (e.g., 'resid 100' or 'name CA and resid 50-60')
        """
        selection = self.universe.select_atoms(selection_string)
        
        if len(selection) == 0:
            raise ValueError(f"Selection '{selection_string}' did not match any atoms")
            
        self.start_point = selection.center_of_mass()
        return self.start_point
    
    def set_start_point_from_pocket(self, ligand_selection, fpocket_min_alpha_sphere=3, max_pocket_distance=5.0):
        """
        Set the tunnel starting point as the center of a pocket containing a specified selection.
        Uses fpocket or similar methodology to identify pockets.
        
        Parameters:
        -----------
        ligand_selection : str
            MDAnalysis selection string for ligand or region of interest
        fpocket_min_alpha_sphere : int
            Minimum number of alpha spheres to define a pocket
        max_pocket_distance : float
            Maximum distance (Å) between selection and pocket center to consider it "containing" the selection
            
        Returns:
        --------
        numpy.ndarray
            Coordinates of the pocket center
        """
        # Identify the ligand/selection of interest
        selection = self.universe.select_atoms(ligand_selection)
        
        if len(selection) == 0:
            raise ValueError(f"Selection '{ligand_selection}' did not match any atoms")
        
        selection_center = selection.center_of_mass()
        protein_selection = self.universe.select_atoms("protein")

        # Implementation of a simplified pocket detection algorithm
        # This is inspired by fpocket but implemented directly to avoid external dependencies
        pockets = self.internal_pocket_detection(protein_selection)
        
        # Find the closest pocket to the selection center
        closest_pocket = None
        min_distance = float('inf')
        
        for pocket in pockets:
            distance = np.linalg.norm(pocket['center'] - selection_center)
            if distance < min_distance and distance <= max_pocket_distance:
                min_distance = distance
                closest_pocket = pocket
        
        if closest_pocket is None:
            raise ValueError(f"No pocket found containing the selection '{ligand_selection}'")
        
        self.start_point = closest_pocket['center']
        return self.start_point
    
    def internal_pocket_detection(self, protein_selection):
        """
        Simple internal pocket detection algorithm based on alpha spheres.
        This is a simplified version of fpocket's methodology.
        """
        pockets = []
        protein_positions = protein_selection.positions
        protein_radii = np.array([self.VDW_RADII.get(atom.element.upper(), self.VDW_RADII['DEFAULT']) 
                                 for atom in protein_selection])
        
        # Use KDTree for faster nearest neighbor searches
        protein_tree = KDTree(protein_positions)
        
        # Use the Voronoi diagram to find potential alpha sphere centers
        vor = Voronoi(protein_positions)
        
        # Vectorized operations for alpha sphere detection
        alpha_spheres = []
        batch_size = 1000  # Process vertices in batches for memory efficiency
        
        for i in range(0, len(vor.vertices), batch_size):
            batch_vertices = vor.vertices[i:i+batch_size]
            
            # Find distances from vertices to atoms using KDTree for speed
            distances, indices = protein_tree.query(batch_vertices, k=4)
            
            # Calculate minimum distances considering atom radii
            min_distances = distances - protein_radii[indices]
            min_dist_per_vertex = np.min(min_distances, axis=1)
            max_dist_per_vertex = np.max(distances, axis=1)
            
            # Find vertices that qualify as alpha spheres
            valid_mask = (min_dist_per_vertex > 0) & (min_dist_per_vertex < 3.0) & (max_dist_per_vertex < 10.0)
            valid_indices = np.where(valid_mask)[0]
            
            for j in valid_indices:
                alpha_spheres.append({
                    'center': batch_vertices[j],
                    'radius': min_dist_per_vertex[j],
                    'nearest_atoms': indices[j]
                })
        
        # Step 2: Cluster alpha spheres into pockets
        if not alpha_spheres:
            return pockets
        
        # Faster clustering using KDTree
        if len(alpha_spheres) > 0:
            centers = np.array([sphere['center'] for sphere in alpha_spheres])
            center_tree = KDTree(centers)
            
            # Find clusters using radius neighbor query
            clusters = []
            already_clustered = set()
            
            for i, sphere in enumerate(alpha_spheres):
                if i in already_clustered:
                    continue
                    
                # Find all neighbors within clustering distance
                neighbors = center_tree.query_ball_point(sphere['center'], 3.0)
                
                # Create a new cluster
                new_cluster = [sphere]
                already_clustered.add(i)
                
                # Add neighbors to cluster
                for neighbor_idx in neighbors:
                    if neighbor_idx != i and neighbor_idx not in already_clustered:
                        new_cluster.append(alpha_spheres[neighbor_idx])
                        already_clustered.add(neighbor_idx)
                
                clusters.append(new_cluster)
        
        # Step 3: Convert clusters to pockets
        for cluster in clusters:
            if len(cluster) >= 3:  # Minimum alpha spheres to define a pocket
                centers = [sphere['center'] for sphere in cluster]
                pockets.append({
                    'center': np.mean(centers, axis=0),
                    'alpha_spheres': centers,
                    'size': len(centers)
                })
        
        return pockets
    
    def build_initial_voronoi(self):
        """Build the initial Voronoi diagram based on atom centers."""
        self.voronoi = Voronoi(self.positions)
        return self.voronoi
    
    def enhance_voronoi_with_surface_points(self):
        """
        Enhance the Voronoi diagram by adding points on vdW surface 
        in the direction of neighboring vertices.
        """
        # Initialize storage for enhanced diagram data
        surface_points = []
        surface_atom_indices = []
        
        unit_directions = np.array([[1, 0, 0],[0,1,0],[0,0,1]])

        #for each atom add surface points
        for i, (pos, radius) in enumerate(zip(self.positions, self.atom_radii)):
            
            # Create points on the vdW surface
            surface_points_pos = pos + unit_directions * radius
            surface_points_neg = pos - unit_directions * radius
            new_surface_points = np.vstack((surface_points_pos, surface_points_neg))
            
            #add points to graph
            for surface_point in new_surface_points:
                surface_points.append(surface_point)
                surface_atom_indices.append(i)
        
        # Add surface points to the positions
        enhanced_positions = np.array(surface_points)
        self.enhanced_positions = enhanced_positions
            
        # Rebuild Voronoi diagram with enhanced positions
        self.enhanced_voronoi = Voronoi(enhanced_positions)
        
        # Store mapping from surface points to their parent atoms
        self.surface_to_atom = {}
        for i, idx in enumerate(surface_atom_indices):
            self.surface_to_atom[i + len(self.positions)] = idx
            
        return self.enhanced_voronoi
    
    def build_tunnel_graph(self):
        """
        Build a graph representing the network of tunnels through the protein.
        This then needs to be post processed to be used for dijkstra
        """
        self.graph = nx.Graph()
        
        vertices = self.enhanced_voronoi.vertices

        # Add nodes to graph
        for i, vertex in enumerate(vertices):
            self.graph.add_node(i, position=vertex)
        
        #add edges to graph
        for ridge in self.enhanced_voronoi.ridge_vertices:
            if ridge[0] != -1 and ridge[1] != -1:  # -1 means infinity in Voronoi diagram                 
                    vertex1 = vertices[ridge[0]]
                    vertex2 = vertices[ridge[1]]
                    distance = np.linalg.norm(vertex1 - vertex2)
                    self.graph.add_edge(ridge[0], ridge[1], distance=distance)

        return self.graph

    def vdw_filter_graph(self, length_filter = 4):
        """
        Filter out Voronoi vertices that are inside atom vdW radii.
        Also removes points very far from the protein

        length_filter = number of rgyrs away from protein that we care about vertices

        """
        
        # Use KDTree for more efficient near-neighbor queries
        self.atom_tree = KDTree(self.positions)
        atom_tree = self.atom_tree
        com = self.protein_com
        rgyr = self.protein_rgyr
        # Precompute positions and atom radii to avoid repeated attribute accesses
        node_positions = np.array(list(nx.get_node_attributes(self.graph, "position").values()))  # Get all vertex positions at once
        atom_radii = self.atom_radii
        vdw_removal_counter = 0
        dist_removal_counter = 0

        for i, vertex in enumerate(node_positions):

            #check if vertex is close to protein
            if np.linalg.norm(vertex - com) > rgyr * length_filter:
                self.graph.remove_node(i)
                dist_removal_counter += 1

            else:
                # Get the neighbors for this vertex via ball point
                max_radius = np.max(atom_radii)
                neighbors = atom_tree.query_ball_point(vertex, max_radius + 1.0)

                # Check if vertex is inside any atom's vdW radius
                for j in neighbors:
                    dist = np.linalg.norm(vertex - self.positions[j])

                    if np.any(dist < (atom_radii[j] + self.min_bottleneck)):
                        self.graph.remove_node(i)
                        vdw_removal_counter += 1
                        break  # No need to check further once a node is removed
        
        # Generate a new node ID mapping
        new_node_ids = {old_id: new_id for new_id, old_id in enumerate(self.graph.nodes)}

        #Reassign the node IDs in the graph
        self.graph = nx.relabel_nodes(self.graph, new_node_ids)

        #Store positions of valid nodes in a NP array
        #graph to np conversion
        positions = nx.get_node_attributes(self.graph, "position")

        # Convert the dictionary of positions to a 2D NumPy array
        self.graph_node_positions = np.array(list(positions.values()))

        print(f"removed {vdw_removal_counter} verticies for vdw reasons")
        print(f"removed {dist_removal_counter} verticies for being to far from protein")

        return self.graph
    
    def calc_rmax(self, thresh = 1e-6):
        """
        Calculates Rmax for all points in the graph
        """
        atom_tree = self.atom_tree
        negative_rmax = 0

        for node in self.graph.nodes:
            # Use KDTree for more efficient near-neighbor queries
            node_position = self.graph.nodes[node]["position"]

            #check the five neraest neighboring atoms for a clash
            distances, neighbors_index = self.atom_tree.query(node_position, k=5)

            rmax = np.min(distances - self.atom_radii[neighbors_index])

            #no negative weights
            if rmax > thresh:
                self.graph.nodes[node]["rmax"] = rmax
            else:
                negative_rmax += 1
                self.graph.nodes[node]["rmax"] = 0
                self.graph.nodes[node]["valid"] = None

        print(f"found {negative_rmax} points with negative rmax")
        return self.graph

    
    def calc_graph_weights(self):
        """
        calculates weight for each edge using a linear approximation
        weight = distance^2 / Rmaxavg
        """

        for node1, node2 in self.graph.edges(data=False):
            self.graph.edges[node1, node2]["weight"] = (2 * self.graph.edges[node1, node2]["distance"])/(self.graph.nodes[node1]["rmax"] + self.graph.nodes[node2]["rmax"])
            
        return self.graph

    def bulk_vertices_check(self):
        "checks if a graph node is outside the convex hull"

        # Calculate convex hull for the protein from the vdw radii positions
        self.hull = ConvexHull(self.enhanced_positions)
        self.bulk_vertices_index = []

        # Create a Delaunay triangulation of the hull points for faster point-in-hull check
        hull_triangulation = Delaunay(self.hull.points)

        #classify bulk and nonbulk
        for i, vertex in enumerate(self.graph_node_positions):
            if hull_triangulation.find_simplex(vertex) < 0:
                self.bulk_vertices_index.append(i)

        unclassified_idx = range(len(self.graph.nodes)).remove(self.bulk_vertices_index)
        search_bulk_idx = self.bulk_vertices_index
        node_tree = KDTree(self.graph_node_positions)

        #TODO implement advancing
        #start advancing into the protein
        while True:
            # Find all points within the search radius of the current bulk points
            new_bulk_indices = []

            for idx in search_bulk_idx:
                neighbors = node_tree.query_ball_point(self.graph.nodes[idx], self.probe_radius)

                #For nonbulk neighbors check if halfway distance rmax is smaller than self.probe_radii
                #And distance between points is less than 5
                #then assign bulk
                for neighbor in filter(lambda n: n not in search_bulk_idx, neighbors):


            # Remove already classified bulk points
            new_bulk_indices.difference_update(np.where(search_bulk_idx)[0])
            

            if not new_bulk_indices:
                break  # No new points to add, exit the loop

            #add newly assigned bulk to search_bulk_idx
            bulk_mask[list(new_bulk_indices)] = True

        return self.bulk_vertices_index   


    def _find_closest_vertex_to_point(self, point):
        """Find the closest valid Voronoi vertex to a given point."""
        
        # Calculate distances and find minimum
        distances = np.linalg.norm(self.graph_node_positions - point, axis=1)
        closest_idx = np.argmin(distances)
        
        return closest_idx

    def find_tunnels(self, start_point=None, max_tunnels=10):
        # Determine the starting point
        if start_point is not None:
            self.start_point = np.array(start_point)
        elif self.start_point is None:
            raise ValueError("No starting point provided or previously set")

        # Find the closest valid vertex to the starting point
        self.start_vertex = self._find_closest_vertex_to_point(self.start_point)
        print(f"Start vertex: {self.start_vertex}")

        # Prepare tunnel data storage
        self.tunnels = []
        tunnel_data = []

        # Run Dijkstra’s algorithm to find shortest paths from the start vertex
        lengths, paths = nx.single_source_dijkstra(self.graph, self.start_vertex, weight='weight')
        # Filter paths to only those ending at bulk solvent vertices
        bulk_vertices = set(self.bulk_vertices_index)
        valid_tunnels = []

        for bulk_idx in bulk_vertices:
            if bulk_idx in lengths:
                path = paths[bulk_idx]

                # Compute actual path length (sum of physical distances)
                path_length = sum(self.graph[path[i]][path[i + 1]]['distance'] for i in range(len(path) - 1))

                # Store total weight from Dijkstra (used for throughput calculation)
                total_weight = lengths[bulk_idx]

                valid_tunnels.append((bulk_idx, total_weight, path_length, np.hstack(path)))
                
        # Sort tunnels by highest throughput (1 / total_weight)
        valid_tunnels.sort(key=lambda x: x[1], reverse=False)  # lower weight first

        # Select the 10 best tunnels (or fewer if not enough exist)
        for i, (bulk_idx, total_weight, path_length, path) in enumerate(valid_tunnels[:max_tunnels]):
            # Extract path details
            path_vertices = np.vstack([self.graph_node_positions[idx] for idx in path])
            path_rmax = [self.graph.nodes[idx]['rmax'] for idx in path]

            # Check bottleneck radius condition
            min_rmax = min(path_rmax)
            if min_rmax >= self.min_bottleneck:
                self.tunnels.append({
                    'path_indices': path,
                    'path_vertices': path_vertices,
                    'path_rmax': path_rmax,
                    'length': path_length,  # Physical distance
                    'total_weight': total_weight,  # Dijkstra weight
                    'throughput': 1 / total_weight,  # Higher is better
                    'bottleneck': min_rmax,
                    'bottleneck_idx': path_rmax.index(min_rmax)
                })

                # Store tunnel data for analysis
                tunnel_data.append({
                    'tunnel_id': len(self.tunnels),
                    'length': path_length,  # Physical length
                    'total_weight': total_weight,  # Dijkstra weight
                    'throughput': 1 / total_weight,  # Higher is better
                    'bottleneck': min_rmax,
                    'bottleneck_position': path_vertices[path_rmax.index(min_rmax)],
                    'start': path_vertices[0],
                    'end': path_vertices[-1],
                    'num_points': len(path),
                    'start_point': self.start_point
                })

        # Create DataFrame with tunnel data
        self.tunnel_dataframe = pd.DataFrame(tunnel_data) if tunnel_data else pd.DataFrame()

        return self.tunnel_dataframe 


    def save_tunnels_to_pdb(self, filename='tunnels.pdb'):
        """
        Save tunnels as different models in a single PDB file.
        Rmax values are written to the B-factor column.
        """
        if not self.tunnels:
            raise ValueError("No tunnels found. Run find_tunnels() first.")
        
        with open(filename, 'w') as f:
            for i, tunnel in enumerate(self.tunnels):
                f.write(f"MODEL {i+1}\n")
                f.write(f"REMARK   1 TUNNEL {i+1} LENGTH {tunnel['length']:.2f} BOTTLENECK {tunnel['bottleneck']:.2f}\n")
                
                for j, (vertex, rmax) in enumerate(zip(tunnel['path_vertices'], tunnel['path_rmax'])):
                    x, y, z = vertex
                    atom_name = "O" if j == tunnel['bottleneck_idx'] else "C"
                    f.write(f"ATOM  {j+1:5d} {atom_name:4s} PTH X{i+1:4d}    {x:8.3f}{y:8.3f}{z:8.3f}  1.00{rmax:6.2f}           {atom_name}\n")
                
                f.write("ENDMDL\n")
        
        print(f"Tunnels saved to {filename}")


    def analyze_protein_tunnels(self, start_method, max_tunnels=10, 
                            output_pdb='tunnels.pdb', custom_coords=None, 
                            selection_string=None, 
                            n_processes=None):
        """
        Complete pipeline for protein tunnel analysis.
        
        Parameters:
        -----------
        start_method : str
            Method to determine starting point: 'coordinates', 'selection', or 'pocket'
        max_tunnels : int
            Maximum number of tunnels to find
        min_bottleneck : float
            Minimum bottleneck radius (Å) for a valid tunnel
        output_pdb : str
            Filename for saving tunnels as PDB
        custom_coords : array-like
            Custom coordinates for starting point (used if start_method='coordinates')
        selection_string : str
            MDAnalysis selection string (used if start_method='selection')
        ligand_selection : str
            MDAnalysis selection string for ligand or region of interest (used if start_method='pocket')
        n_processes : int or None
            Number of processes to use for multiprocessing.
            If None, will use number of available CPU cores.
            
        Returns:
        --------
        pd.DataFrame
            DataFrame containing tunnel information
        """

        print(f"atoms {len(self.positions)}")
        print("Building initial Voronoi diagram...")
        voronoi = self.build_initial_voronoi()
        print(f"Found {len(self.voronoi.vertices)} valid vertices without enhancing\n")
        
        print("Enhancing Voronoi diagram with surface points...")
        #self.enhanced_voronoi = voronoi
        self.enhance_voronoi_with_surface_points()
        print(f"Found {len(self.enhanced_voronoi.vertices)} valid vertices with vdw enhancing\n")

        print("Building tunnel graph...")
        self.build_tunnel_graph()
        print(f"edges = {len(self.graph.edges())}")
        print(f"nodes = {len(self.graph.nodes())}\n")

        print("Vdw filtering graph...")
        self.vdw_filter_graph(length_filter=3)
        print(f"edges = {len(self.graph.edges())}")
        print(f"nodes = {len(self.graph.nodes())}\n")

        print("calculating rmax")
        self.calc_rmax()

        print("calculating weights")
        self.calc_graph_weights()

        print("finding bulk vertices")
        self.bulk_vertices_check()
        print(f"Found {len(self.bulk_vertices_index)} bulk verticies\n") 
        

        # Set starting point based on method
        if start_method == 'coordinates':
            if custom_coords is None:
                raise ValueError("Custom coordinates must be provided when start_method='coordinates'")
            self.set_start_point_from_coordinates(custom_coords)
        elif start_method == 'selection':
            if selection_string is None:
                raise ValueError("Selection string must be provided when start_method='selection'")
            self.set_start_point_from_selection(selection_string)
        elif start_method == 'pocket':
            if selection_string is None:
                raise ValueError("Selection string must be provided when start_method='pocket'")
            self.set_start_point_from_pocket(selection_string)
        else:
            raise ValueError(f"Unknown start_method: {start_method}")
        
        print(f"Finding tunnels from starting point: {self.start_point}")
        tunnels_df = self.find_tunnels(max_tunnels=max_tunnels)
        
        if len(tunnels_df) > 0:
            self.save_tunnels_to_pdb(filename=output_pdb)
            print(f"Found {len(tunnels_df)} tunnels with bottleneck >= {self.min_bottleneck}Å")
        else:
            print(f"No tunnels found with bottleneck >= {self.min_bottleneck}Å")
        
        return tunnels_df


# Example usage
if __name__ == "__main__":
    # Load protein structure with MDAnalysis
    #pdb_file = "1cqw-prep.pdb"
    pdb_file = "../1be0.pdb"
    pdb_name = pdb_name = os.path.splitext(pdb_file)[0]
    u = mda.Universe(pdb_file)
    
    analyzer1 = ProteinTunnelAnalyzer(u, selection='protein', probe_radius=3, min_bottleneck = 0)
    #tunnels_df1 = analyzer1.analyze_protein_tunnels("pocket", selection_string="resid 117 or resid 187 or resid 183" , 
    #                                                max_tunnels=10, output_pdb=f"{pdb_file}_tunnels.pdb", n_processes=22)
    tunnels_df1 = analyzer1.analyze_protein_tunnels("selection", selection_string="resname ACY" , 
                                                    max_tunnels=10, output_pdb=f"{pdb_file}_tunnels.pdb", n_processes=22)
    print(tunnels_df1.head)
    
    if False:
        # Example 1: Using custom coordinates
        analyzer2 = ProteinTunnelAnalyzer(u, selection='protein', probe_radius=3.0)
        custom_coords = [10.0, 15.0, 20.0]  # Example coordinates
        tunnels_df2 = analyzer2.analyze_protein_tunnels(max_tunnels=10, min_bottleneck=1.0,
                                                    output_pdb="tunnels_custom.pdb",
                                                    start_method='coordinates',
                                                    custom_coords=custom_coords)
        
        # Example 2: Using selection center
        analyzer3 = ProteinTunnelAnalyzer(u, selection='protein', probe_radius=3.0)
        tunnels_df3 = analyzer3.analyze_protein_tunnels(max_tunnels=10, min_bottleneck=1.0,
                                                    output_pdb="tunnels_selection.pdb",
                                                    start_method='selection',
                                                    selection_string='resid 100 and name CA')
        
        # Example 3: Using pocket detection
        analyzer4 = ProteinTunnelAnalyzer(u, selection='protein', probe_radius=3.0)
        tunnels_df4 = analyzer4.analyze_protein_tunnels(max_tunnels=10, min_bottleneck=1.0,
                                                    output_pdb="tunnels_pocket.pdb",
                                                    start_method='pocket',
                                                    ligand_selection='resname LIG')

atoms 2478
Building initial Voronoi diagram...
Found 16189 valid vertices without enhancing

Enhancing Voronoi diagram with surface points...
Found 16189 valid vertices with vdw enhancing

Building tunnel graph...
edges = 15015
nodes = 16189

Vdw filtering graph...
removed 1784 verticies for vdw reasons
removed 127 verticies for being to far from protein
edges = 12259
nodes = 14278

calculating rmax
found 0 points with negative rmax
calculating weights
finding bulk vertices
Found 1636 bulk verticies

Finding tunnels from starting point: [27.38661536 31.6406086  30.44637707]
Start vertex: 12190
No tunnels found with bottleneck >= 0Å
<bound method NDFrame.head of Empty DataFrame
Columns: []
Index: []>


In [5]:
def write_pdb_from_array(filename, coords, atom_name="X", res_name="UNK", chain_id="A", res_seq=1):
    """
    Write a NumPy array of 3D coordinates to a PDB file.
    
    Parameters:
    -----------
    filename : str
        Name of the output PDB file.
    coords : np.ndarray
        NumPy array of shape (N, 3) containing x, y, z coordinates.
    atom_name : str
        Name of the atom (default: "X").
    res_name : str
        Residue name (default: "UNK").
    chain_id : str
        Chain identifier (default: "A").
    res_seq : int
        Residue sequence number (default: 1).
    """
    with open(filename, "w") as f:
        for i, (x, y, z) in enumerate(coords, start=1):
            f.write(f"ATOM  {i:5d} {atom_name:<4} {res_name} {chain_id}{res_seq:4d}    {x:8.3f}{y:8.3f}{z:8.3f}\n")
        f.write("END\n")  # PDB termination line

def write_pdb_from_array_list(filename, array_list, atom_name="C", res_name="UNK", chain_id="A", res_seq=1):
    """
    Write a NumPy array of 3D coordinates to a PDB file.
    
    Parameters:
    -----------
    filename : str
        Name of the output PDB file.
    coords : np.ndarray
        NumPy array of shape (N, 3) containing x, y, z coordinates.
    atom_name : str
        Name of the atom (default: "X").
    res_name : str
        Residue name (default: "UNK").
    chain_id : str
        Chain identifier (default: "A").
    res_seq : int
        Residue sequence number (default: 1).
    """
    with open(filename, "w") as f:
        for j, array in enumerate(array_list):
            f.write(f"MODEL {j+1}\n")
            for i, (x, y, z) in enumerate(np.atleast_2d(array), start=1):
                f.write(f"ATOM  {i:5d} {atom_name:<4} {res_name} {chain_id}{res_seq:4d}    {x:8.2f}{y:8.2f}{z:8.2f}\n")
            if j == 2:
                temp = list(analyzer1.graph.edges)
                for edge in temp:
                    f.write(f"CONECT{int(edge[0]+1):5d}{int(edge[1]+1):5d}\n")
            f.write("ENDMDL\n")
        f.write("END\n")  # PDB termination line

#write_pdb_from_array("temp.pdb", analyzer1.filtered_voronoi_vertices[analyzer1.bulk_vertices])

analysis_list=[analyzer1.graph_node_positions[analyzer1.bulk_vertices_index], analyzer1.graph_node_positions[analyzer1.start_vertex], analyzer1.graph_node_positions]

write_pdb_from_array_list("temp_list.pdb", analysis_list)

In [37]:

array = ConvexHull(analyzer1.positions).vertices
write_pdb_from_array("convex.pdb", analyzer1.positions[array])
write_pdb_from_array("positions.pdb", analyzer1.positions)

In [83]:
import py3Dmol

def visualize_tunnel_graph(graph, scale=0.5, color="blue"):
    """
    Visualize the tunnel graph stored in a NetworkX graph object using py3Dmol.
    
    Parameters:
    - graph: NetworkX graph where nodes contain 'position' (3D coordinates).
    - scale: Scaling factor for sphere and bond sizes.
    - color: Color of the bonds (edges).
    
    Returns:
    - py3Dmol view object.
    """
    view = py3Dmol.view(width=800, height=600)
    
    # Add nodes as spheres
    for node, data in graph.nodes(data=True):
        x, y, z = data["position"]
        view.addSphere({
            "center": {"x": x, "y": y, "z": z},
            "radius": 0.3 * scale,
            "color": "red",
            "opacity": 0.8
        })
    
    # Add edges as cylinders
    for u, v, edge_data in graph.edges(data=True):
        x1, y1, z1 = graph.nodes[u]["position"]
        x2, y2, z2 = graph.nodes[v]["position"]
        
        view.addCylinder({
            "start": {"x": x1, "y": y1, "z": z1},
            "end": {"x": x2, "y": y2, "z": z2},
            "radius": 0.15 * scale,
            "color": color,
            "opacity": 0.6
        })
    
    view.zoomTo()
    return view

# Example usage:
view = visualize_tunnel_graph(analyzer1.graph)
view.show()



KeyboardInterrupt: 

In [ ]:
#TODO implement advancing into protein from hull
#TODO connect disconnected points inside protein by creating nodes and edges at the halfway mark


[[-60.44931239  46.35476963  44.14339846]
 [ -3.07698498  -1.94652627  63.37304315]
 [ 46.3641732   37.70793907  63.75069554]
 ...
 [ 25.43882387  43.05582716  26.69591786]
 [ 25.21102903  42.59074152  27.99706938]
 [ 26.18101921  42.30137125  26.97634457]]


In [76]:
edge_weights = np.array([analyzer1.graph.edges[edge]["weight"] for edge in analyzer1.graph.edges])
print(np.min(edge_weights))


node_rmax = np.array([analyzer1.graph.nodes[node]["rmax"] for node in analyzer1.graph.nodes])
print(np.min(node_rmax))

TypeError: '<=' not supported between instances of 'float' and 'NoneType'

0
